In [ ]:
import requests
import base64
import io
from PIL import Image as PILImage
from dotenv import load_dotenv
from tools.models_utils import score_aesthetic
from tools.exif_tool import fetch_exif
from tools.captioning_tool import caption_image
from rag.retriever_fetch_tool import retrieve_photography_tips
from agent.agent_state import ToolCalls
from langchain.chat_models import init_chat_model
from agent.graph import build_graph
from IPython.display import Image, display

load_dotenv()

response_model = init_chat_model("gpt-5-nano", temperature=0)
tool_decider_model = response_model.with_structured_output(ToolCalls)

graph = build_graph(tool_decider_model, response_model, response_model)

mermaid_str = graph.get_graph().draw_mermaid()
mermaid_str = mermaid_str.replace("graph TD;", "graph LR;")

# Render via mermaid.ink
encoded = base64.urlsafe_b64encode(mermaid_str.encode()).decode()
response = requests.get(f"https://mermaid.ink/img/{encoded}?type=png")
response.raise_for_status()

# Flatten onto white background
img = PILImage.open(io.BytesIO(response.content)).convert("RGBA")
white_bg = PILImage.new("RGBA", img.size, (255, 255, 255, 255))
white_bg.paste(img, mask=img.split()[3])
final = white_bg.convert("RGB")
final.save("flowchart.png")

display(Image("flowchart.png"))